# KWISMO — Notebook 01 : Analyse Exploratoire des Données (EDA)

Ce notebook effectue l'analyse statistique et visuelle du jeu de données KWISMO.

### 📌 Compatibilité Multi-Sources & Environnements :
- Fonctionne en **Local**, sur **Google Colab** et sur **Kaggle Notebooks**.
- Recherche automatique du dataset depuis **Local**, **Google Drive** (`/content/drive/MyDrive/kwismo_data`) ou **Git**.

In [ ]:
# 1. Détection d'environnement et Localisation du Dataset (Local / Drive / Git)
import os
import sys
import json
from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

# Chemins de recherche possibles pour le dataset (dans l'ordre de priorité)
POSSIBLE_PATHS = [
    Path.cwd() / "data" / "processed" / "model_b_clean.jsonl",
    Path.cwd() / "data" / "raw" / "kwismo_data" / "messages.jsonl",
    Path.cwd() / "data" / "raw" / "scraped" / "messages.jsonl",
    Path("/content/drive/MyDrive/kwismo_data/messages.jsonl"),
    Path("/kaggle/working/kwismo/kwismo-ai/data/processed/model_b_clean.jsonl"),
    Path("/content/kwismo/kwismo-ai/data/processed/model_b_clean.jsonl")
]

dataset_path = None
for p in POSSIBLE_PATHS:
    if p.exists():
        dataset_path = p
        break

if dataset_path:
    print(f"✅ Dataset trouvé avec succès : {dataset_path}")
else:
    print("⚠️ Dataset introuvable dans les emplacements standard. Recherche dans le répertoire courant...")
    dataset_path = Path.cwd()

## 2. Chargement et Statistiques Globales

In [ ]:
import pandas as pd
import numpy as np

records = []
if dataset_path and dataset_path.is_file():
    with open(dataset_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"Nombre total de lignes chargées : {len(df)}")
if not df.empty:
    print(df.head())
else:
    print("Le DataFrame est vide.")

## 3. Analyse de la Longueur des Textes (Mots & Caractères)

In [ ]:
if not df.empty and "texte" in df.columns:
    df["longueur_caracteres"] = df["texte"].fillna("").apply(len)
    df["nombre_mots"] = df["texte"].fillna("").apply(lambda t: len(t.split()))
    
    print("=== Statistiques sur le nombre de mots ===")
    print(df["nombre_mots"].describe())
    
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 4))
    plt.hist(df["nombre_mots"], bins=30, color="#2FAC66", edgecolor="black")
    plt.title("Distribution du nombre de mots par extrait de fraude")
    plt.xlabel("Nombre de mots")
    plt.ylabel("Fréquence")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.show()

## 4. Analyse des Mots-Clés Récurrents (N-grams & Termes de Fraude)

In [ ]:
from collections import Counter
import re

if not df.empty and "texte" in df.columns:
    all_words = []
    stopwords = {"de", "la", "le", "les", "des", "un", "une", "et", "a", "en", "du", "pour", "sur", "est", "pas", "plus", "par", "que", "dans", "avec", "au", "ce", "qui", "ne"}
    
    for text in df["texte"].dropna():
        words = re.findall(r"\b\w+\b", text.lower())
        filtered = [w for w in words if w not in stopwords and len(w) > 2]
        all_words.extend(filtered)
        
    counter = Counter(all_words)
    print("=== Top 20 des mots les plus fréquents ===")
    top20 = counter.most_common(20)
    for word, freq in top20:
        print(f"{word:20s} : {freq}")

## 5. Répartition par Type d'Origine et Source

In [ ]:
if not df.empty:
    if "type_origine" in df.columns:
        print("=== Répartition par Type d'Origine ===")
        print(df["type_origine"].value_counts())
    if "source_origine" in df.columns:
        print("\n=== Répartition par Source ===")
        print(df["source_origine"].value_counts().head(10))